## Imports y Configuración

Controla el bloque, su tamaño y el solapamiento desde esta celda.


In [1]:
from pathlib import Path
import math
import shutil

import cv2
import numpy as np

VIDEO = Path("Videos") / "video30min-11to22.mp4"
SALIDA = Path("Videos")
FPS_SALIDA = 5.0
SEGUNDO_INICIO = None
SEGUNDO_FIN = None

NUMERO_BLOQUE = 1
FRAMES_PRINCIPALES_POR_BLOQUE = 100
SOLAPAMIENTO_TOTAL = 20

EXTRAER_FRAMES_NUEVAMENTE = False
LIMPIAR_BLOQUE_ANTERIOR = True
GUARDAR_DIAGNOSTICOS = True
COMPRESION_PNG = 3


## Extracción de Frames

Usa los frames existentes o vuelve a extraerlos cuando se indique.


In [ ]:
def guardar_png(ruta, imagen):
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(ruta), imagen, [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG]):
        raise RuntimeError(f"No se pudo guardar {ruta}")


def extraer_frames(video=VIDEO, salida=SALIDA, forzar=EXTRAER_FRAMES_NUEVAMENTE):
    video = Path(video)
    base = Path(salida) / video.stem
    carpeta = base / "frames"
    existentes = sorted(carpeta.glob("*.png"))

    if existentes and not forzar:
        print(f"Se usarán {len(existentes)} frames existentes")
        return base

    if not video.is_file():
        raise FileNotFoundError(video)

    carpeta.mkdir(parents=True, exist_ok=True)
    for archivo in carpeta.glob("*.png"):
        archivo.unlink()

    captura = cv2.VideoCapture(str(video))
    fps_video = float(captura.get(cv2.CAP_PROP_FPS))
    total_video = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps_video <= 0 or total_video <= 0:
        captura.release()
        raise RuntimeError("No se pudieron leer los datos del video")

    duracion = total_video / fps_video
    inicio = 0.0 if SEGUNDO_INICIO is None else max(0.0, float(SEGUNDO_INICIO))
    fin = duracion if SEGUNDO_FIN is None else min(duracion, float(SEGUNDO_FIN))
    tiempos = np.arange(inicio, fin, 1.0 / FPS_SALIDA, dtype=np.float64)
    digitos = max(4, len(str(max(0, len(tiempos)-1))))

    guardados = 0
    for indice, tiempo in enumerate(tiempos):
        captura.set(cv2.CAP_PROP_POS_MSEC, float(tiempo) * 1000.0)
        correcto, frame = captura.read()
        if not correcto:
            break
        guardar_png(carpeta / f"{indice:0{digitos}d}.png", frame)
        guardados += 1
        if guardados % 100 == 0:
            print(f"Frames: {guardados}/{len(tiempos)}")

    captura.release()
    print(f"Frames extraídos: {guardados}")
    return base

## Plan de Bloques

Divide el solapamiento total en 10 frames a cada lado cuando el valor es 20.


In [3]:
def calcular_bloques(total_frames, frames_principales, solapamiento_total):
    izquierda = solapamiento_total // 2
    derecha = solapamiento_total - izquierda
    cantidad = math.ceil(total_frames / frames_principales)
    bloques = []

    for indice in range(cantidad):
        inicio = indice * frames_principales
        fin = min(total_frames, inicio + frames_principales)
        bloques.append({
            "numero": indice + 1,
            "inicio": inicio,
            "fin": fin,
            "inicio_contexto": max(0, inicio - izquierda),
            "fin_contexto": min(total_frames, fin + derecha),
        })

    return bloques


def mostrar_plan(base, frames_principales=FRAMES_PRINCIPALES_POR_BLOQUE):
    frames = sorted((Path(base) / "frames").glob("*.png"))
    bloques = calcular_bloques(len(frames), frames_principales, SOLAPAMIENTO_TOTAL)
    print(f"Frames disponibles: {len(frames)}")
    print(f"Bloques necesarios: {len(bloques)}")

    for bloque in bloques:
        principales = bloque["fin"] - bloque["inicio"]
        total = bloque["fin_contexto"] - bloque["inicio_contexto"]
        print(f"Bloque {bloque['numero']:02d}: {principales} principales, {total} para procesar")

    return bloques


## Configuración del HUD

Usa detección estricta y márgenes pequeños para evitar manchas grandes.


In [4]:
VERDE_FUERTE = (37, 90, 100, 50, 14, 11)
VERDE_MEDIO = (32, 100, 45, 30, 7, 6)
VERDE_TENUE = (29, 106, 22, 20, 3, 3)

MUESTRAS_ATLAS = 300
UMBRAL_ATLAS = 0.10
RADIO_ATLAS = 4
DILATACION_TEXTO = 2
DILATACION_BRUJULA = 2

DX_INICIAL = 0.080
DY_INICIAL = 0.082
DX_MINIMO = 0.060
DX_MAXIMO = 0.125
DY_MINIMO = 0.055
DY_MAXIMO = 0.145
PASO_BUSQUEDA = 2

GROSOR_CROSSHAIR = 5
MARGEN_CROSSHAIR = 2
RADIO_RESIDUAL_CROSSHAIR = 5

GROSOR_CIRCULO = 7
MARGEN_CIRCULO = 2
TOLERANCIA_ARCO = 4


## Detección del HUD

Detecta el verde y restringe cada componente a su zona conocida.


In [5]:
def detectar_verde(frame, configuracion):
    h1, h2, s1, v1, dr, db = configuracion
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    azul, verde, rojo = cv2.split(frame)
    azul = azul.astype(np.int16)
    verde = verde.astype(np.int16)
    rojo = rojo.astype(np.int16)

    rango = cv2.inRange(
        hsv,
        np.array([h1, s1, v1], np.uint8),
        np.array([h2, 255, 255], np.uint8),
    )
    dominio = ((verde >= rojo + dr) & (verde >= azul + db)).astype(np.uint8) * 255
    return cv2.bitwise_and(rango, dominio)


def crear_zona(forma, cajas):
    alto, ancho = forma
    mascara = np.zeros(forma, np.uint8)
    for x1, y1, x2, y2 in cajas:
        cv2.rectangle(
            mascara,
            (round(ancho*x1), round(alto*y1)),
            (round(ancho*x2), round(alto*y2)),
            255,
            -1,
        )
    return mascara


def obtener_zonas(forma):
    texto = crear_zona(forma, [
        (.010,.005,.285,.170), (.010,.170,.115,.400),
        (.720,.005,.995,.175), (.005,.610,.095,.750),
        (.005,.770,.285,.998), (.835,.735,.998,.998),
    ])
    brujula = crear_zona(forma, [(.375,.000,.625,.170)])
    crosshair = crear_zona(forma, [(.330,.260,.670,.770)])
    inferior = crear_zona(forma, [(.380,.800,.620,.998)])
    return texto, brujula, crosshair, inferior


def obtener_zonas_circulares(forma):
    return [
        crear_zona(forma, [(.405,.825,.500,.998)]),
        crear_zona(forma, [(.495,.825,.590,.998)]),
    ]


def expandir(mascara, radio):
    tamano = 2*int(radio)+1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tamano, tamano))
    return cv2.dilate(mascara, kernel)


## Reconstrucción del Crosshair

Detecta las esquinas en L y reconstruye únicamente las líneas conocidas.


In [6]:
def evidencia_crosshair(frame, zona_crosshair):
    fuerte = cv2.bitwise_and(detectar_verde(frame, VERDE_FUERTE), zona_crosshair)
    medio = cv2.bitwise_and(detectar_verde(frame, VERDE_MEDIO), zona_crosshair)
    tenue = cv2.bitwise_and(detectar_verde(frame, VERDE_TENUE), zona_crosshair)
    anclas = cv2.bitwise_or(fuerte, medio)
    tenue = cv2.bitwise_and(tenue, expandir(anclas, 4))
    return cv2.bitwise_or(anclas, tenue)


def dibujar_esquinas(forma, dx, dy, grosor):
    alto, ancho = forma
    cx, cy = ancho//2, alto//2
    mascara = np.zeros(forma, np.uint8)
    largo_h = max(18, int(ancho*.026))
    largo_v = max(18, int(alto*.036))

    for sx, sy in [(-1,-1),(1,-1),(-1,1),(1,1)]:
        x, y = cx+sx*dx, cy+sy*dy
        cv2.line(mascara, (x,y), (x-sx*largo_h,y), 255, grosor, cv2.LINE_AA)
        cv2.line(mascara, (x,y), (x,y-sy*largo_v), 255, grosor, cv2.LINE_AA)
    return mascara


def apoyo_esquinas(evidencia, dx, dy):
    alto, ancho = evidencia.shape
    cx, cy = ancho//2, alto//2
    soporte = expandir(evidencia, 2) > 0
    largo_h = max(18, int(ancho*.026))
    largo_v = max(18, int(alto*.036))
    apoyos = []

    for sx, sy in [(-1,-1),(1,-1),(-1,1),(1,1)]:
        x, y = cx+sx*dx, cy+sy*dy
        h = np.zeros_like(evidencia)
        v = np.zeros_like(evidencia)
        cv2.line(h, (x,y), (x-sx*largo_h,y), 255, 3)
        cv2.line(v, (x,y), (x,y-sy*largo_v), 255, 3)
        ph, pv = h>0, v>0
        ah = np.count_nonzero(ph & soporte) / max(1, np.count_nonzero(ph))
        av = np.count_nonzero(pv & soporte) / max(1, np.count_nonzero(pv))
        apoyos.append(min(ah, av))

    buenas = sum(a >= .12 for a in apoyos)
    puntaje = np.median(apoyos)*260 + np.mean(apoyos)*80 + buenas*22
    return float(puntaje), buenas, apoyos


def estimar_apertura(evidencia, anterior=None):
    alto, ancho = evidencia.shape
    if anterior is None:
        anterior = (int(ancho*DX_INICIAL), int(alto*DY_INICIAL))

    mejor = anterior
    mejor_puntaje, mejores, apoyos = apoyo_esquinas(evidencia, *anterior)

    for dy in range(int(alto*DY_MINIMO), int(alto*DY_MAXIMO)+1, PASO_BUSQUEDA):
        for dx in range(int(ancho*DX_MINIMO), int(ancho*DX_MAXIMO)+1, PASO_BUSQUEDA):
            puntaje, buenas, actuales = apoyo_esquinas(evidencia, dx, dy)
            puntaje -= .025*(abs(dx-anterior[0]) + abs(dy-anterior[1]))
            if puntaje > mejor_puntaje:
                mejor, mejor_puntaje, mejores, apoyos = (dx,dy), puntaje, buenas, actuales

    if mejores < 2:
        return anterior, mejores, mejor_puntaje, apoyos

    dx = round(.45*anterior[0] + .55*mejor[0])
    dy = round(.45*anterior[1] + .55*mejor[1])
    return (int(round(dx/2)*2), int(round(dy/2)*2)), mejores, mejor_puntaje, apoyos


def dibujar_crosshair(forma, apertura):
    alto, ancho = forma
    cx, cy = ancho//2, alto//2
    dx, dy = apertura
    mascara = dibujar_esquinas(forma, dx, dy, GROSOR_CROSSHAIR)

    radio_x = 10
    cv2.line(mascara, (cx-radio_x,cy-radio_x), (cx+radio_x,cy+radio_x), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)
    cv2.line(mascara, (cx-radio_x,cy+radio_x), (cx+radio_x,cy-radio_x), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)

    brazo_h = max(20, int(dx*.72))
    brazo_v = max(14, int(dy*.56))
    cv2.line(mascara, (cx-brazo_h,cy), (cx+brazo_h,cy), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)
    cv2.line(mascara, (cx,cy-brazo_v), (cx,cy+brazo_v), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)

    desplazamiento = int(dx*.80)
    largo = max(7, int(dy*.16))
    for x in (cx-desplazamiento, cx+desplazamiento):
        cv2.line(mascara, (x,cy-largo), (x,cy+largo), 255, GROSOR_CROSSHAIR, cv2.LINE_AA)
    return mascara


def crear_mascara_crosshair(frame, zona_crosshair, anterior=None):
    evidencia = evidencia_crosshair(frame, zona_crosshair)
    apertura, esquinas, puntaje, apoyos = estimar_apertura(evidencia, anterior)
    mascara = expandir(dibujar_crosshair(evidencia.shape, apertura), MARGEN_CROSSHAIR)

    residual = cv2.bitwise_and(evidencia, expandir(mascara, RADIO_RESIDUAL_CROSSHAIR))
    residual = cv2.bitwise_and(residual, expandir(mascara, 2))
    mascara = cv2.bitwise_or(mascara, expandir(residual, 1))
    mascara = cv2.bitwise_and(mascara, zona_crosshair)
    return mascara, apertura, esquinas, puntaje, apoyos, evidencia


## Reconstrucción de Círculos

Busca el centro y el radio que mejor coinciden con el arco y dibuja solo el contorno.


In [7]:
def puntuar_anillo(xs, ys, cx, cy, radio):
    distancias = np.sqrt((xs-cx)**2 + (ys-cy)**2)
    error = np.abs(distancias-radio)
    coincidencias = np.count_nonzero(error <= TOLERANCIA_ARCO)
    cercanas = np.count_nonzero(error <= TOLERANCIA_ARCO+3)
    return coincidencias*3 + cercanas - float(np.median(error))


def detectar_circulo(frame, zona_circulo, anterior=None):
    fuerte = detectar_verde(frame, VERDE_FUERTE)
    medio = detectar_verde(frame, VERDE_MEDIO)
    señal = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), zona_circulo)

    yz, xz = np.where(zona_circulo > 0)
    x1, x2 = int(xz.min()), int(xz.max())+1
    y1, y2 = int(yz.min()), int(yz.max())+1
    ys, xs = np.where(señal[y1:y2, x1:x2] > 0)

    if len(xs) < 10:
        return anterior, señal, 0.0

    xs = xs.astype(np.float64)+x1
    ys = ys.astype(np.float64)+y1
    esperado_x = (x1+x2)/2
    esperado_y = (y1+y2)/2
    ancho_total = señal.shape[1]
    radio_min = max(18, int(ancho_total*.018))
    radio_max = max(radio_min+2, int(ancho_total*.048))

    mejor = anterior
    mejor_puntaje = -1e9

    for cy in range(int(esperado_y-30), int(esperado_y+31), 2):
        for cx in range(int(esperado_x-25), int(esperado_x+26), 2):
            distancias = np.sqrt((xs-cx)**2 + (ys-cy)**2)
            for radio in range(radio_min, radio_max+1, 2):
                puntaje = puntuar_anillo(xs, ys, cx, cy, radio)
                puntaje -= .035*(abs(cx-esperado_x)+abs(cy-esperado_y))
                if anterior is not None:
                    puntaje -= .025*(abs(cx-anterior[0])+abs(cy-anterior[1])+abs(radio-anterior[2]))
                if puntaje > mejor_puntaje:
                    mejor, mejor_puntaje = (cx,cy,radio), puntaje

    if mejor is None:
        return anterior, señal, mejor_puntaje

    if anterior is not None:
        mejor = tuple(int(round(.70*a+.30*n)) for a,n in zip(anterior, mejor))
    return mejor, señal, mejor_puntaje


def reconstruir_circulo(forma, parametros):
    mascara = np.zeros(forma, np.uint8)
    if parametros is None:
        return mascara
    cx, cy, radio = parametros
    cv2.circle(
        mascara,
        (cx,cy),
        radio+MARGEN_CIRCULO,
        255,
        GROSOR_CIRCULO,
        cv2.LINE_AA,
    )
    return mascara


## Máscaras del Bloque

Crea máscaras precisas, vistas y diagnósticos solamente para el bloque elegido.


In [8]:
def crear_atlas_texto(frames):
    cantidad = min(MUESTRAS_ATLAS, len(frames))
    indices = np.linspace(0, len(frames)-1, cantidad).astype(int)
    acumulador = None

    for indice in indices:
        frame = cv2.imread(str(frames[indice]))
        fuerte = detectar_verde(frame, VERDE_FUERTE)
        zona_texto, _, _, _ = obtener_zonas(fuerte.shape)
        muestra = cv2.bitwise_and(fuerte, zona_texto)
        if acumulador is None:
            acumulador = np.zeros(muestra.shape, np.float32)
        acumulador += (muestra>0).astype(np.float32)

    return ((acumulador/cantidad)>=UMBRAL_ATLAS).astype(np.uint8)*255


def crear_bloque(base, numero_bloque, frames_principales):
    base = Path(base)
    todos = sorted((base/"frames").glob("*.png"))
    bloques = calcular_bloques(len(todos), frames_principales, SOLAPAMIENTO_TOTAL)

    if numero_bloque < 1 or numero_bloque > len(bloques):
        raise ValueError(f"El bloque debe estar entre 1 y {len(bloques)}")

    datos = bloques[numero_bloque-1]
    carpeta = base/f"bloque_{numero_bloque:02d}"
    if carpeta.exists() and LIMPIAR_BLOQUE_ANTERIOR:
        shutil.rmtree(carpeta)

    for nombre in ("frames","masks","vistas","diagnosticos","no_hud"):
        (carpeta/nombre).mkdir(parents=True, exist_ok=True)

    frames = todos[datos["inicio_contexto"]:datos["fin_contexto"]]
    atlas = crear_atlas_texto(todos)
    guardar_png(carpeta/"atlas_texto.png", atlas)

    apertura = None
    circulos = [None,None]

    for indice, ruta in enumerate(frames):
        frame = cv2.imread(str(ruta))
        fuerte = detectar_verde(frame, VERDE_FUERTE)
        medio = detectar_verde(frame, VERDE_MEDIO)
        tenue = detectar_verde(frame, VERDE_TENUE)
        z_texto, z_brujula, z_crosshair, z_inferior = obtener_zonas(fuerte.shape)

        soporte_texto = cv2.bitwise_and(expandir(atlas, RADIO_ATLAS), z_texto)
        m_texto = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), soporte_texto)
        m_texto = expandir(m_texto, DILATACION_TEXTO)

        m_cross, apertura, esquinas, puntaje, apoyos, evidencia = crear_mascara_crosshair(
            frame, z_crosshair, apertura
        )

        m_brujula = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), z_brujula)
        m_brujula = cv2.morphologyEx(
            m_brujula,
            cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_RECT,(3,7)),
        )
        m_brujula = expandir(m_brujula, DILATACION_BRUJULA)

        m_inferior = cv2.bitwise_and(cv2.bitwise_or(fuerte, medio), z_inferior)
        m_inferior = expandir(m_inferior, 2)

        zonas_circulos = obtener_zonas_circulares(fuerte.shape)
        mascaras_circulos = []
        señales_circulos = []
        puntajes_circulos = []

        for i, zona_circulo in enumerate(zonas_circulos):
            circulos[i], señal, puntaje_circulo = detectar_circulo(frame, zona_circulo, circulos[i])
            mascaras_circulos.append(reconstruir_circulo(fuerte.shape, circulos[i]))
            señales_circulos.append(señal)
            puntajes_circulos.append(puntaje_circulo)

        mascara = cv2.bitwise_or(m_texto, m_cross)
        mascara = cv2.bitwise_or(mascara, m_brujula)
        mascara = cv2.bitwise_or(mascara, m_inferior)
        for m_circulo in mascaras_circulos:
            mascara = cv2.bitwise_or(mascara, m_circulo)

        zonas = cv2.bitwise_or(z_texto, z_brujula)
        zonas = cv2.bitwise_or(zonas, z_crosshair)
        zonas = cv2.bitwise_or(zonas, z_inferior)
        residual = cv2.bitwise_and(tenue, zonas)
        residual = cv2.bitwise_and(residual, expandir(mascara, 3))
        mascara = cv2.bitwise_or(mascara, expandir(residual, 1))
        mascara = (mascara>0).astype(np.uint8)*255

        shutil.copy2(ruta, carpeta/"frames"/ruta.name)
        guardar_png(carpeta/"masks"/ruta.name, mascara)

        rojo = np.zeros_like(frame)
        rojo[:,:,2] = 255
        mezcla = cv2.addWeighted(frame,.65,rojo,.35,0)
        vista = frame.copy()
        vista[mascara>0] = mezcla[mascara>0]
        guardar_png(carpeta/"vistas"/ruta.name, vista)

        if GUARDAR_DIAGNOSTICOS:
            diagnostico = frame.copy()
            plantilla = dibujar_crosshair(mascara.shape, apertura)
            diagnostico[plantilla>0] = (255,0,255)

            for parametros in circulos:
                if parametros is not None:
                    cx,cy,radio = parametros
                    cv2.circle(diagnostico,(cx,cy),radio,(0,255,255),2,cv2.LINE_AA)
                    cv2.circle(diagnostico,(cx,cy),3,(255,255,0),-1)

            texto = f"dx={apertura[0]} dy={apertura[1]} esquinas={esquinas} apoyos={[round(a,2) for a in apoyos]}"
            cv2.putText(diagnostico,texto,(20,30),cv2.FONT_HERSHEY_SIMPLEX,.55,(255,255,0),2,cv2.LINE_AA)
            guardar_png(carpeta/"diagnosticos"/ruta.name, diagnostico)

        if (indice+1)%25 == 0:
            print(f"Bloque {numero_bloque:02d}: {indice+1}/{len(frames)}")

    nombres = [todos[i].name for i in range(datos["inicio"],datos["fin"])]
    (carpeta/"frames_finales.txt").write_text("\n".join(nombres)+"\n",encoding="utf-8")
    print(f"Bloque listo: {carpeta}")
    return carpeta


## Probar el Bloque Elegido

Cambia NUMERO_BLOQUE o FRAMES_PRINCIPALES_POR_BLOQUE en la primera celda y ejecuta todo.


In [ ]:
def preparar_todos_los_bloques(
    base,
    frames_principales=500,
    solapamiento_total=20,
):
    global SOLAPAMIENTO_TOTAL

    solapamiento_anterior = SOLAPAMIENTO_TOTAL
    SOLAPAMIENTO_TOTAL = solapamiento_total

    frames = sorted(
        (Path(base) / "frames").glob("*.png")
    )

    bloques = calcular_bloques(
        total_frames=len(frames),
        frames_principales=frames_principales,
        solapamiento_total=solapamiento_total,
    )

    print(f"Frames disponibles: {len(frames)}")
    print(f"Bloques por crear: {len(bloques)}")
    print(f"Frames principales por bloque: {frames_principales}")
    print(f"Solapamiento total: {solapamiento_total}")
    print()

    carpetas = []

    try:
        for bloque in bloques:
            numero = bloque["numero"]

            print(
                f"Procesando bloque {numero:02d} "
                f"de {len(bloques):02d}"
            )

            carpeta = crear_bloque(
                base=base,
                numero_bloque=numero,
                frames_principales=frames_principales,
            )

            carpetas.append(carpeta)

    finally:
        SOLAPAMIENTO_TOTAL = solapamiento_anterior

    print()
    print("Todos los bloques quedaron listos.")

    return carpetas


base = SALIDA / VIDEO.stem

carpetas_bloques = preparar_todos_los_bloques(
    base=base,
    frames_principales=500,
    solapamiento_total=20,
)

Frames disponibles: 3300
Bloques por crear: 7
Frames principales por bloque: 500
Solapamiento total: 20

Procesando bloque 01 de 07
Bloque 01: 25/510
Bloque 01: 50/510
Bloque 01: 75/510
Bloque 01: 100/510
Bloque 01: 125/510
Bloque 01: 150/510
Bloque 01: 175/510
Bloque 01: 200/510
Bloque 01: 225/510
Bloque 01: 250/510
Bloque 01: 275/510
Bloque 01: 300/510
Bloque 01: 325/510
Bloque 01: 350/510
Bloque 01: 375/510
Bloque 01: 400/510
Bloque 01: 425/510
Bloque 01: 450/510
Bloque 01: 475/510
Bloque 01: 500/510
Bloque listo: Videos\video30min-11to22\bloque_01
Procesando bloque 02 de 07
Bloque 02: 25/520
Bloque 02: 50/520
Bloque 02: 75/520
Bloque 02: 100/520
Bloque 02: 125/520
Bloque 02: 150/520
Bloque 02: 175/520
Bloque 02: 200/520
Bloque 02: 225/520
Bloque 02: 250/520
Bloque 02: 275/520
Bloque 02: 300/520
Bloque 02: 325/520
Bloque 02: 350/520
Bloque 02: 375/520
Bloque 02: 400/520
Bloque 02: 425/520
Bloque 02: 450/520
Bloque 02: 475/520
Bloque 02: 500/520
Bloque listo: Videos\video30min-11to22

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path


def contar_png_resultantes(carpeta_salida):
    carpeta_salida = Path(carpeta_salida)

    salida_directa = list(
        carpeta_salida.glob("*.png")
    )

    salida_anidada = list(
        (carpeta_salida / "frames").glob("*.png")
    )

    return max(
        len(salida_directa),
        len(salida_anidada),
    )


def ejecutar_propainter_en_bloques(
    base,
    repo_propainter=Path("ProPainter"),
    omitir_terminados=True,
    detener_si_falla=True,
):
    base = Path(base).resolve()
    repo_propainter = Path(repo_propainter).resolve()
    python_propainter = Path(sys.executable).resolve()

    script_propainter = (
        repo_propainter
        / "inference_propainter.py"
    )

    if not base.is_dir():
        raise FileNotFoundError(
            f"No existe la carpeta del video: {base}"
        )

    if not repo_propainter.is_dir():
        raise FileNotFoundError(
            f"No existe la carpeta ProPainter: {repo_propainter}"
        )

    if not script_propainter.is_file():
        raise FileNotFoundError(
            f"No existe el script: {script_propainter}"
        )

    if not python_propainter.is_file():
        raise FileNotFoundError(
            f"No existe el Python activo: {python_propainter}"
        )

    bloques = sorted(
        carpeta
        for carpeta in base.glob("bloque_*")
        if carpeta.is_dir()
    )

    if not bloques:
        raise RuntimeError(
            f"No se encontraron bloques en {base}"
        )

    print("Comprobando el entorno de ProPainter...")
    print(f"Python activo: {python_propainter}")

    comprobacion = subprocess.run(
        [
            str(python_propainter),
            "-c",
            (
                "import torch; "
                "print('PyTorch:', torch.__version__); "
                "print('CUDA:', torch.cuda.is_available()); "
                "print('GPU:', torch.cuda.get_device_name(0) "
                "if torch.cuda.is_available() else 'Sin GPU')"
            ),
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(comprobacion.stdout)

    if comprobacion.returncode != 0:
        print(comprobacion.stderr)

        raise RuntimeError(
            "No fue posible importar PyTorch en el entorno activo"
        )

    if "CUDA: True" not in comprobacion.stdout:
        raise RuntimeError(
            "El entorno activo no tiene CUDA disponible. "
            "Selecciona el kernel Python - ProPainter."
        )

    entorno = os.environ.copy()

    entorno["PYTHONNOUSERSITE"] = "1"
    entorno["PYTORCH_CUDA_ALLOC_CONF"] = (
        "expandable_segments:True"
    )
    entorno["CUDA_VISIBLE_DEVICES"] = "0"
    entorno["PYTHONUNBUFFERED"] = "1"

    print(f"Bloques encontrados: {len(bloques)}")
    print(f"Repositorio: {repo_propainter}")
    print()

    completados = []
    omitidos = []
    fallidos = []

    for posicion, bloque in enumerate(
        bloques,
        start=1,
    ):
        carpeta_frames = (
            bloque
            / "frames"
        )

        carpeta_masks = (
            bloque
            / "masks"
        )

        carpeta_salida = (
            bloque
            / "no_hud"
        )

        frames = sorted(
            carpeta_frames.glob("*.png")
        )

        mascaras = sorted(
            carpeta_masks.glob("*.png")
        )

        cantidad_frames = len(
            frames
        )

        cantidad_mascaras = len(
            mascaras
        )

        cantidad_resultados = (
            contar_png_resultantes(
                carpeta_salida
            )
        )

        print()
        print(
            f"Bloque {posicion}/{len(bloques)}: "
            f"{bloque.name}"
        )

        print(
            f"Frames: {cantidad_frames}"
        )

        print(
            f"Máscaras: {cantidad_mascaras}"
        )

        print(
            f"Resultados existentes: "
            f"{cantidad_resultados}"
        )

        if cantidad_frames == 0:
            print(
                "Omitido: no contiene frames."
            )

            omitidos.append(
                bloque.name
            )

            continue

        if cantidad_mascaras != cantidad_frames:
            print(
                "Error: la cantidad de máscaras "
                "no coincide con la de frames."
            )

            fallidos.append(
                bloque.name
            )

            if detener_si_falla:
                break

            continue

        nombres_frames = [
            archivo.name
            for archivo in frames
        ]

        nombres_mascaras = [
            archivo.name
            for archivo in mascaras
        ]

        if nombres_frames != nombres_mascaras:
            print(
                "Error: los nombres de frames "
                "y máscaras no coinciden."
            )

            fallidos.append(
                bloque.name
            )

            if detener_si_falla:
                break

            continue

        if (
            omitir_terminados
            and cantidad_resultados
            == cantidad_frames
        ):
            print(
                "Omitido: el bloque ya está completo."
            )

            omitidos.append(
                bloque.name
            )

            continue

        if carpeta_salida.exists():
            print(
                "Eliminando salida incompleta anterior..."
            )

            shutil.rmtree(
                carpeta_salida
            )

        carpeta_salida.mkdir(
            parents=True,
            exist_ok=True,
        )

        comando = [
            str(python_propainter),
            "-s",
            "-u",
            "inference_propainter.py",
            "--video",
            str(carpeta_frames.resolve()),
            "--mask",
            str(carpeta_masks.resolve()),
            "--output",
            str(carpeta_salida.resolve()),
            "--width",
            "1280",
            "--height",
            "720",
            "--fp16",
            "--save_frames",
            "--neighbor_length",
            "10",
            "--ref_stride",
            "10",
            "--subvideo_length",
            "40",
            "--mask_dilation",
            "3",
            "--raft_iter",
            "20",
        ]

        print(
            "Iniciando ProPainter..."
        )

        print(
            " ".join(
                comando
            )
        )

        print()

        proceso = subprocess.Popen(
            comando,
            cwd=str(repo_propainter),
            env=entorno,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )

        if proceso.stdout is None:
            proceso.kill()

            raise RuntimeError(
                "No fue posible leer la salida de ProPainter"
            )

        for linea in proceso.stdout:
            print(
                f"[{bloque.name}] {linea}",
                end="",
                flush=True,
            )

        codigo_salida = proceso.wait()

        if codigo_salida != 0:
            print()

            print(
                f"Error en {bloque.name}. "
                f"Código de salida: {codigo_salida}"
            )

            fallidos.append(
                bloque.name
            )

            if detener_si_falla:
                break

            continue

        cantidad_resultados = (
            contar_png_resultantes(
                carpeta_salida
            )
        )

        if cantidad_resultados != cantidad_frames:
            print()

            print(
                f"Salida incompleta: se esperaban "
                f"{cantidad_frames} resultados y se "
                f"encontraron {cantidad_resultados}."
            )

            fallidos.append(
                bloque.name
            )

            if detener_si_falla:
                break

            continue

        print()

        print(
            f"{bloque.name} terminado correctamente: "
            f"{cantidad_resultados} resultados."
        )

        completados.append(
            bloque.name
        )

    print()
    print("Resumen de ProPainter")
    print(f"Completados: {len(completados)}")
    print(f"Omitidos: {len(omitidos)}")
    print(f"Fallidos: {len(fallidos)}")

    if completados:
        print(
            "Completados:",
            ", ".join(
                completados
            ),
        )

    if omitidos:
        print(
            "Omitidos:",
            ", ".join(
                omitidos
            ),
        )

    if fallidos:
        print(
            "Fallidos:",
            ", ".join(
                fallidos
            ),
        )

    return {
        "completados": completados,
        "omitidos": omitidos,
        "fallidos": fallidos,
    }


resultado_propainter = ejecutar_propainter_en_bloques(
    base=SALIDA / VIDEO.stem,
    repo_propainter=Path("ProPainter"),
    omitir_terminados=True,
)